# Stratified longshot exploration

Slices `strat_bucket_stats` by series, distance bucket, and price bucket.

In [ ]:
import os, sqlite3
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

DB = Path(os.environ.get('PREDMARKBOT_RESEARCH_DB',
    str(Path.home() / '.local/share/predmarkbot/research.db')))
conn = sqlite3.connect(f'file:{DB}?mode=ro', uri=True)

In [ ]:
strat = pd.read_sql('SELECT * FROM strat_bucket_stats', conn)
strat.groupby(['series_ticker', 'horizon']).size().unstack(fill_value=0)

In [ ]:
# Heatmap for one series at T-6h
series = strat.series_ticker.value_counts().index[0]
sub = strat[(strat.series_ticker==series) & (strat.horizon=='T-6h')]
pivot = sub.pivot_table(index='price_bucket_lo', columns='distance_bucket_idx', values='bias_bps', aggfunc='first')
sns.heatmap(pivot, cmap='RdBu_r', center=0, annot=True, fmt='.0f', cbar_kws={'label': 'bias (bps)'})
plt.title(f'{series} · T-6h · bias (bps)')

In [ ]:
# Top 20 cells by absolute bias (n>=50)
strat[(strat.n_markets>=50) & (strat.p_value<0.01)].reindex(
    strat[(strat.n_markets>=50) & (strat.p_value<0.01)].bias_bps.abs().sort_values(ascending=False).index
)[['horizon','series_ticker','price_bucket_lo','distance_bucket_idx','n_markets','bias_bps','p_value']].head(20)